Exploratory Data Analysis - Federeated Learning and Agentic AI for NHS UK, Operations Research

In [1]:
import sys , platform , datetime as dt
import pandas as pd , numpy as np


print(dt.datetime.now().strftime("%Y-%m-%d %H:%M:%S"), "|", platform.node())

print("username : Nilesh")

print("python", sys.version.split()[0], "| pandas", pd.__version__, "| numpy", np.__version__)

2026-07-11 03:13:09 | ubuntu-Standard-PC-Q35-ICH9-2009
username : Nilesh
python 3.10.12 | pandas 2.3.3 | numpy 2.2.6


In [2]:
import os , shutil , subprocess , platform 

print("System Specifications")

cpu = platform.processor()
if shutil.which("lscpu"):                                            # lscpu gives the proper model name
    out = subprocess.run(["lscpu"], capture_output=True, text=True).stdout
    cpu = next((l.split(":",1)[1].strip() for l in out.splitlines()
                if l.startswith("Model name")), cpu)                 # grab the "Model name" line
print(f"CPU: {cpu} | {os.cpu_count()} cores")   

ram_gb = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1e9
print(f"RAM: {ram_gb:.0f} GB")

if shutil.which("nvidia-smi"):
    q = subprocess.run(["nvidia-smi",
 "--query-gpu=name,memory.total,memory.used",
 "--format=csv,noheader"], capture_output=True, text=True)
    print("GPU:" , q.stdout.strip())


System Specifications
CPU: Intel Xeon Processor (Cascadelake) | 42 cores
RAM: 67 GB
GPU: NVIDIA A100 80GB PCIe, 81920 MiB, 1 MiB


In [3]:
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "CLAUDE.md").exists())
DATA = ROOT / "data" / "nhs_rtt_referral_to_treatment"
files = sorted(DATA.glob("rtt_*.csv"))

print(len(files), "files:", files[0].name, "→", files[-1].name) 
sizes = [f.stat().st_size/1e6 for f in files]
print(f"MB: min {min(sizes):.0f} max {max(sizes):.0f}")
assert len(files) == 96  

96 files: rtt_2018-04.csv → rtt_2026-03.csv
MB: min 42 max 114


In [4]:
print("Sentinel Months placed as markers")
Sentinels = ["2018-04", "2020-04", "2021-04", "2026-03"]       # sentinels: oldest, COVID crash, mid, newest
df = {m: pd.read_csv(DATA/f"rtt_{m}.csv",              # read each sentinel into a dict of DataFrames
                     low_memory=False) for m in Sentinels}     # low_memory=False: infer dtypes from whole file
for m in Sentinels: print(m, df[m].shape)                      # rows × columns per era (expect 69 vs 121 cols)

Sentinel Months placed as markers


2018-04 (196340, 69)
2020-04 (125460, 69)
2021-04 (160251, 121)
2026-03 (183400, 121)


In [5]:
print("Canonical Months that are most useful")
CANON = ["Period","Provider Org Code","Provider Org Name",        # the 9 columns the loader
         "Treatment Function Code","Treatment Function Name",     # will depend on
         "RTT Part Type","Total All"]

for m in S:
    print(m, "missing:", [c for c in CANON if c not in df[m].columns])  # any canonical col absent?
    print("  parent cols:", [c for c in df[m].columns if "Parent" in c]) # exact Provider Parent headers

Canonical Months that are most useful


NameError: name 'S' is not defined